# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EgeGln365/FlyRank_AI_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")


if HF_TOKEN is None:
    raise ValueError("HF_TOKEN bulunamadı. .env dosyanı kontrol et.")


In [2]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [4]:
april_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {TABLES["fact_daily"]}
    WHERE report_date BETWEEN DATE '2026-04-01'
                          AND DATE '2026-04-30'
""").df()

april_check

,rows,clients,contents,min_date,max_date
0,10424730,61,362172,2026-04-01,2026-04-30


In [5]:
march_april_overlap = con.sql(f"""
WITH march AS (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM {TABLES["fact_daily"]}
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'
),

april AS (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM {TABLES["fact_daily"]}
    WHERE report_date BETWEEN DATE '2026-04-01'
                          AND DATE '2026-04-30'
)

SELECT
    COUNT(*) AS march_contents,

    COUNT(*) FILTER (
        WHERE april.content_hash_id IS NOT NULL
    ) AS matched_in_april,

    COUNT(*) FILTER (
        WHERE april.content_hash_id IS NULL
    ) AS missing_in_april,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE april.content_hash_id IS NOT NULL
        ) / COUNT(*),
        2
    ) AS match_rate_pct

FROM march
LEFT JOIN april
    USING (client_hash_id, content_hash_id)
""").df()

march_april_overlap

,march_contents,matched_in_april,missing_in_april,match_rate_pct
0,331437,331436,1,100.0


In [6]:
april_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) AS observed_days,

        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_days

    FROM {TABLES["fact_daily"]}

    WHERE report_date BETWEEN DATE '2026-04-01'
                          AND DATE '2026-04-30'

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

april_coverage["gsc_available_days"].describe()

count    362172.000000
mean         10.771291
std          12.830818
min           0.000000
25%           0.000000
50%           2.000000
75%          27.000000
max          30.000000
Name: gsc_available_days, dtype: float64

In [7]:
for days in [1, 7, 14, 20, 27, 30]:
    n = (april_coverage["gsc_available_days"] >= days).sum()
    pct = 100 * n / len(april_coverage)

    print(
        f">= {days:2d} days: "
        f"{n:,} contents ({pct:.2f}%)"
    )

>=  1 days: 194,760 contents (53.78%)
>=  7 days: 155,300 contents (42.88%)
>= 14 days: 132,907 contents (36.70%)
>= 20 days: 115,929 contents (32.01%)
>= 27 days: 93,265 contents (25.75%)
>= 30 days: 74,572 contents (20.59%)


In [8]:
april_content = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_april,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_april,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS ctr_april,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_april

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2026-04-01'
                      AND DATE '2026-04-30'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [9]:
april_content[
    april_content["gsc_available_days"] >= 20
][
    [
        "impressions_april",
        "clicks_april",
        "ctr_april",
        "avg_position_april"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
impressions_april,115929.0,2469.041922,7211.541135,21.000000,167.00000,535.000000,1974.000000,799358.000000
clicks_april,115929.0,6.990425,36.336944,0.000000,0.00000,1.000000,4.000000,7434.000000
ctr_april,115929.0,0.002212,0.004270,0.000000,0.00000,0.000368,0.002876,0.144444
avg_position_april,115929.0,16.996083,15.708146,0.019608,5.94401,10.543253,23.501859,114.017241


In [10]:
april_target_data = april_content[
    (april_content["gsc_available_days"] >= 20) &
    (april_content["impressions_april"] > 0) &
    (april_content["avg_position_april"] > 0) &
    (april_content["ctr_april"].notna())
].copy()

In [11]:
import pandas as pd

april_target_data["position_bucket"] = pd.cut(
    april_target_data["avg_position_april"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "21+"],
    include_lowest=True
)

In [12]:
april_ctr_by_position = (
    april_target_data[
        april_target_data["ctr_april"] > 0
    ]
    .groupby(
        "position_bucket",
        observed=True
    )["ctr_april"]
    .quantile([0.10, 0.25, 0.50])
    .unstack()
)

april_ctr_by_position.columns = [
    "p10_ctr",
    "p25_ctr",
    "median_ctr"
]

april_ctr_by_position

,p10_ctr,p25_ctr,median_ctr
position_bucket,,,
1-3,0.000673,0.001348,0.002928
4-10,0.000807,0.001505,0.002907
11-20,0.000919,0.001656,0.003143
21+,0.000408,0.000864,0.001890


In [13]:
april_target_data["position_bucket_fine"] = pd.cut(
    april_target_data["avg_position_april"],
    bins=[
        0,2,4,6,8,10,
        15,20,30, float("inf")
    ],
    labels=[
        "0-2",
        "2-4",
        "4-6",
        "6-8",
        "8-10",
        "10-15",
        "15-20",
        "20-30",
        "30+"
    ],
    include_lowest=True
)

In [14]:
fine_position_audit = (
    april_target_data
    .groupby(
        "position_bucket_fine",
        observed=True
    )
    .agg(
        n=("content_hash_id", "size"),
        median_position=("avg_position_april", "median"),
        median_ctr=("ctr_april", "median"),
        mean_ctr=("ctr_april", "mean"),
        median_impressions=("impressions_april", "median")
    )
    .reset_index()
)

fine_position_audit

,position_bucket_fine,n,median_position,median_ctr,mean_ctr,median_impressions
0,0-2,1867,1.509977,0.000357,0.001985,865.0
1,2-4,9817,3.233812,0.002172,0.003517,1518.0
2,4-6,17805,5.100000,0.002086,0.003277,1424.0
3,6-8,15729,6.925418,0.001077,0.002628,748.0
4,8-10,10589,8.891605,0.000537,0.002410,517.0
5,10-15,15226,12.146724,0.000000,0.002301,412.0
6,15-20,10084,17.356491,0.000000,0.002050,370.0
7,20-30,14672,24.473528,0.000000,0.001484,401.0
8,30+,20140,41.751299,0.000000,0.000768,202.0


In [15]:
fine_positive_ctr = (
    april_target_data[
        april_target_data["ctr_april"] > 0
    ]
    .groupby(
        "position_bucket_fine",
        observed=True
    )
    .agg(
        n_positive=("content_hash_id", "size"),
        p25_positive_ctr=("ctr_april", lambda x: x.quantile(0.25)),
        median_positive_ctr=("ctr_april", "median")
    )
    .reset_index()
)

fine_positive_ctr

,position_bucket_fine,n_positive,p25_positive_ctr,median_positive_ctr
0,0-2,984,0.001003,0.002185
1,2-4,7296,0.001727,0.003352
2,4-6,13396,0.001656,0.003091
3,6-8,9819,0.001268,0.002513
4,8-10,5661,0.001372,0.002770
5,10-15,7231,0.001695,0.003204
6,15-20,4567,0.001581,0.003040
7,20-30,6235,0.001095,0.002188
8,30+,4751,0.000643,0.001456


In [16]:
march_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days_march,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_march,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS ctr_march,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
        ELSE NULL
    END AS avg_position_march

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2026-03-01'
                      AND DATE '2026-03-31'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [17]:
march_features[
    [
        "gsc_available_days_march",
        "impressions_march",
        "clicks_march",
        "ctr_march",
        "avg_position_march"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
gsc_available_days_march,331437.0,10.895166,13.197774,0.0,0.000000,1.000000,28.000000,31.0
impressions_march,176738.0,1587.986675,5431.337724,1.0,20.000000,173.000000,1039.000000,617124.0
clicks_march,176738.0,4.650002,26.722649,0.0,0.000000,0.000000,2.000000,5668.0
ctr_march,176738.0,0.004594,0.037760,0.0,0.000000,0.000000,0.002158,1.0
avg_position_march,176738.0,15.992270,18.097575,0.0,4.917879,8.177966,20.254025,309.0


In [18]:
modeling_frame = march_features.merge(
    april_content,
    on=["client_hash_id","content_hash_id"],
    how="inner"
)


print("March rows:", len(march_features))
print("April rows:", len(april_content))
print("Joined rows:", len(modeling_frame))

modeling_frame.head()

March rows: 331437
April rows: 362172
Joined rows: 331436


,client_hash_id,content_hash_id,gsc_available_days_march,impressions_march,clicks_march,ctr_march,avg_position_march,gsc_available_days,impressions_april,clicks_april,ctr_april,avg_position_april
0,client_62f4a7e64f5e0096,content_c6bbcab7cd78c380,31,1140.0,0.0,0.000000,12.949123,30,153.0,0.0,0.0,29.620915
1,client_62f4a7e64f5e0096,content_6d978c91fcf0c680,31,458.0,1.0,0.002183,11.884279,29,171.0,0.0,0.0,9.567251
2,client_62f4a7e64f5e0096,content_89f06671812ca9c7,27,343.0,0.0,0.000000,3.586006,9,12.0,0.0,0.0,23.750000
3,client_62f4a7e64f5e0096,content_d54c0e0ff7061da3,31,469.0,0.0,0.000000,7.046908,23,138.0,0.0,0.0,3.398551
4,client_62f4a7e64f5e0096,content_3fa077d5341ea420,30,203.0,1.0,0.004926,7.413793,23,51.0,0.0,0.0,14.470588


In [19]:
coverage_check = {
    "joined_total": len(modeling_frame),

    "march_20_days": (
        modeling_frame["gsc_available_days_march"] >= 20
    ).sum(),

    "april_20_days": (
        modeling_frame["gsc_available_days"] >= 20
    ).sum(),

    "both_20_days": (
        (modeling_frame["gsc_available_days_march"] >= 20)
        &
        (modeling_frame["gsc_available_days"] >= 20)
    ).sum()
}

coverage_check

{'joined_total': 331436,
 'march_20_days': np.int64(106546),
 'april_20_days': np.int64(114263),
 'both_20_days': np.int64(95633)}

In [20]:
model_data = modeling_frame[
    (modeling_frame["gsc_available_days_march"] >= 20)
    &
    (modeling_frame["gsc_available_days"] >= 20)
].copy()

print("Modeling rows:", len(model_data))

Modeling rows: 95633


In [21]:
april_good_position = model_data[
    (model_data["avg_position_april"] > 0)
    &
    (model_data["avg_position_april"] <= 10)
    &
    (model_data["ctr_april"] > 0)
].copy()

print("Good-position + positive-CTR contents:",
      len(april_good_position))

print(
    april_good_position["ctr_april"]
    .quantile([0.10, 0.25, 0.50, 0.75])
)

Good-position + positive-CTR contents: 31540
0.10    0.000738
0.25    0.001394
0.50    0.002716
0.75    0.004890
Name: ctr_april, dtype: float64


In [22]:
ctr_threshold_april = april_good_position["ctr_april"].quantile(0.25)

print("April CTR threshold:", ctr_threshold_april)

April CTR threshold: 0.001394335515512089


In [23]:
opportunity_mask = (
    (model_data["avg_position_april"] > 0)
    &
    (model_data["avg_position_april"] <= 10)
    &
    (model_data["impressions_april"] >= 500)
    &
    (model_data["ctr_april"] <= ctr_threshold_april)
)

n_positive = opportunity_mask.sum()
n_total = len(model_data)

print("Positive opportunities:", n_positive)
print("Total contents:", n_total)
print("Base rate:", n_positive / n_total)

Positive opportunities: 12273
Total contents: 95633
Base rate: 0.12833436156975103


In [24]:
model_data.head(
)

,client_hash_id,content_hash_id,gsc_available_days_march,impressions_march,clicks_march,ctr_march,avg_position_march,gsc_available_days,impressions_april,clicks_april,ctr_april,avg_position_april
0,client_62f4a7e64f5e0096,content_c6bbcab7cd78c380,31,1140.0,0.0,0.000000,12.949123,30,153.0,0.0,0.000000,29.620915
1,client_62f4a7e64f5e0096,content_6d978c91fcf0c680,31,458.0,1.0,0.002183,11.884279,29,171.0,0.0,0.000000,9.567251
3,client_62f4a7e64f5e0096,content_d54c0e0ff7061da3,31,469.0,0.0,0.000000,7.046908,23,138.0,0.0,0.000000,3.398551
4,client_62f4a7e64f5e0096,content_3fa077d5341ea420,30,203.0,1.0,0.004926,7.413793,23,51.0,0.0,0.000000,14.470588
5,client_62f4a7e64f5e0096,content_fd5c5a7de12fb5fe,31,304.0,0.0,0.000000,40.157895,30,213.0,1.0,0.004695,27.333333


In [25]:
feature_cols = [
    "impressions_march",
    "clicks_march",
    "ctr_march",
    "avg_position_march",
    "gsc_available_days_march"
]

feature_audit = model_data[feature_cols].describe(
    percentiles=[0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
).T

feature_audit["missing"] = model_data[feature_cols].isna().sum()
feature_audit

,count,mean,std,min,1%,10%,25%,50%,75%,90%,99%,max,missing
impressions_march,95633.0,2848.382933,7131.397151,25.000000,50.000000,123.00000,273.000000,820.000000,2662.000000,6730.800000,30076.360000,617124.000000,0
clicks_march,95633.0,8.254389,35.703653,0.000000,0.000000,0.00000,0.000000,1.000000,6.000000,20.000000,106.000000,5668.000000,0
ctr_march,95633.0,0.002548,0.004355,0.000000,0.000000,0.00000,0.000000,0.001143,0.003472,0.006711,0.018921,0.162500,0
avg_position_march,95633.0,14.913485,15.112289,0.015873,1.238778,3.06862,4.882893,8.607304,20.000000,35.447746,70.796477,106.890995,0
gsc_available_days_march,95633.0,29.466764,2.775604,20.000000,20.000000,25.00000,29.000000,31.000000,31.000000,31.000000,31.000000,31.000000,0


In [26]:
model_data["opportunity"] = opportunity_mask.astype(int)

In [27]:
target_feature_audit = (
    model_data
    .groupby("opportunity")[feature_cols]
    .median()
    .T
)

target_feature_audit.columns = [
    "not_opportunity",
    "opportunity"
]

target_feature_audit

,not_opportunity,opportunity
impressions_march,659.000000,2276.000000
clicks_march,1.000000,3.000000
ctr_march,0.001134,0.001165
avg_position_march,10.129615,4.961400
gsc_available_days_march,31.000000,31.000000


In [28]:
model_data[
    feature_cols + ["opportunity"]
].corr()["opportunity"].sort_values(ascending=False)

opportunity                 1.000000
impressions_march           0.105440
gsc_available_days_march    0.071215
clicks_march               -0.021295
ctr_march                  -0.090330
avg_position_march         -0.237366
Name: opportunity, dtype: float64

In [29]:
prev90_features = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(DISTINCT report_date) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_days_prev90,

    SUM(gsc_impressions) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS impressions_prev90,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_prev90,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_clicks) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
    END AS ctr_prev90,

    CASE
        WHEN SUM(gsc_impressions) FILTER (
            WHERE gsc_data_available IS TRUE
        ) > 0
        THEN
            SUM(gsc_sum_position) FILTER (
                WHERE gsc_data_available IS TRUE
            ) * 1.0
            /
            SUM(gsc_impressions) FILTER (
                WHERE gsc_data_available IS TRUE
            )
    END AS avg_position_prev90

FROM {TABLES["fact_daily"]}

WHERE report_date BETWEEN DATE '2025-12-01'
                      AND DATE '2026-02-28'

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

In [30]:
model_with_history = model_data.merge(
    prev90_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print("Model data rows:", len(model_data))
print("After join rows:", len(model_with_history))

print(
    "Matched prev90:",
    model_with_history["gsc_available_days_prev90"].notna().sum()
)

print(
    "Missing prev90:",
    model_with_history["gsc_available_days_prev90"].isna().sum()
)

Model data rows: 95633
After join rows: 95633
Matched prev90: 92226
Missing prev90: 3407


In [31]:
model_with_history["gsc_available_days_prev90"].describe(
    percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
)

count    92226.00000
mean        59.50335
std         34.55799
min          0.00000
10%          8.00000
25%         23.00000
50%         79.00000
75%         90.00000
90%         90.00000
max         90.00000
Name: gsc_available_days_prev90, dtype: float64

In [32]:
for days in [1, 7, 14, 30, 60, 75, 90]:
    n = (model_with_history["gsc_available_days_prev90"] >= days).sum()

    print(
        f">= {days:2d} days: {n:,} "
        f"({n / len(model_with_history):.2%})"
    )

>=  1 days: 87,296 (91.28%)
>=  7 days: 83,548 (87.36%)
>= 14 days: 72,839 (76.17%)
>= 30 days: 66,523 (69.56%)
>= 60 days: 55,355 (57.88%)
>= 75 days: 48,641 (50.86%)
>= 90 days: 34,594 (36.17%)


In [33]:
model_with_history["impressions_per_day_march"] = (
    model_with_history["impressions_march"]
    / model_with_history["gsc_available_days_march"]
)

model_with_history["impressions_per_day_prev90"] = (
    model_with_history["impressions_prev90"]
    / model_with_history["gsc_available_days_prev90"]
)

In [34]:
# Trend / change features
model_with_history["ctr_change"] = (
    model_with_history["ctr_march"]
    - model_with_history["ctr_prev90"]
)

model_with_history["position_change"] = (
    model_with_history["avg_position_march"]
    - model_with_history["avg_position_prev90"]
)

model_with_history["impressions_per_day_change"] = (
    model_with_history["impressions_per_day_march"]
    - model_with_history["impressions_per_day_prev90"]
)

In [35]:
trend_cols = [
    "impressions_per_day_march",
    "impressions_per_day_prev90",
    "ctr_change",
    "position_change",
    "impressions_per_day_change"
]

model_with_history[trend_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
impressions_per_day_march,95633.0,93.805213,233.556763,1.035714,9.290323,27.451613,88.724138,21280.137931
impressions_per_day_prev90,87296.0,69.902026,166.064185,1.000000,6.966292,20.722222,64.250000,5364.800000
ctr_change,87296.0,-0.000565,0.007738,-1.000000,-0.001299,0.000000,0.000504,0.133372
position_change,87296.0,3.044098,9.461671,-143.305882,-0.824972,0.907592,5.685356,88.046377
impressions_per_day_change,87296.0,26.513246,152.156107,-3704.638710,-1.206452,3.892597,22.850020,18304.737931


In [36]:
model_with_history[trend_cols].isna().sum()

impressions_per_day_march        0
impressions_per_day_prev90    8337
ctr_change                    8337
position_change               8337
impressions_per_day_change    8337
dtype: int64

In [37]:
model_with_history["has_prev90_history"] = (
    model_with_history["gsc_available_days_prev90"].fillna(0) > 0
).astype(int)

In [38]:
candidate_features = [
    # Current March state
    "impressions_march",
    "ctr_march",
    "avg_position_march",
    "gsc_available_days_march",

    # Historical state
    "impressions_prev90",
    "ctr_prev90",
    "avg_position_prev90",
    "gsc_available_days_prev90",

    # Normalized visibility
    "impressions_per_day_march",
    "impressions_per_day_prev90",

    # Trend
    "ctr_change",
    "position_change",
    "impressions_per_day_change",

    # History availability
    "has_prev90_history"
]

In [39]:
model_with_history[candidate_features].describe(
    percentiles=[0.01, 0.10, 0.50, 0.90, 0.99]
).T

,count,mean,std,min,1%,10%,50%,90%,99%,max
impressions_march,95633.0,2848.382933,7131.397151,25.000000,50.000000,123.000000,820.000000,6730.800000,30076.360000,617124.000000
ctr_march,95633.0,0.002548,0.004355,0.000000,0.000000,0.000000,0.001143,0.006711,0.018921,0.162500
avg_position_march,95633.0,14.913485,15.112289,0.015873,1.238778,3.068620,8.607304,35.447746,70.796477,106.890995
gsc_available_days_march,95633.0,29.466764,2.775604,20.000000,20.000000,25.000000,31.000000,31.000000,31.000000,31.000000
impressions_prev90,87296.0,4840.922940,13045.586554,1.000000,7.000000,88.000000,1127.000000,11865.500000,53323.800000,480045.000000
ctr_prev90,87296.0,0.003002,0.008095,0.000000,0.000000,0.000000,0.001391,0.007112,0.025000,1.000000
avg_position_prev90,87296.0,12.208444,12.791885,0.000000,0.981903,2.733844,7.584828,27.843247,64.667460,150.000000
gsc_available_days_prev90,92226.0,59.503350,34.557990,0.000000,0.000000,8.000000,79.000000,90.000000,90.000000,90.000000
impressions_per_day_march,95633.0,93.805213,233.556763,1.035714,2.173913,4.433333,27.451613,221.419355,976.054372,21280.137931
impressions_per_day_prev90,87296.0,69.902026,166.064185,1.000000,1.384615,3.205128,20.722222,172.116667,703.419444,5364.800000


In [40]:
model_with_history[
    candidate_features + ["opportunity"]
].corr(numeric_only=True)["opportunity"].sort_values(ascending=False)

opportunity                   1.000000
impressions_per_day_march     0.106605
impressions_march             0.105440
impressions_per_day_prev90    0.104735
impressions_prev90            0.097125
gsc_available_days_march      0.071215
gsc_available_days_prev90     0.063828
impressions_per_day_change    0.050753
ctr_change                    0.009585
has_prev90_history           -0.019403
ctr_prev90                   -0.054391
ctr_march                    -0.090330
position_change              -0.135647
avg_position_prev90          -0.186290
avg_position_march           -0.237366
Name: opportunity, dtype: float64

In [41]:
feature_corr = model_with_history[candidate_features].corr(
    numeric_only=True
)

feature_corr.round(2)

,impressions_march,ctr_march,avg_position_march,gsc_available_days_march,impressions_prev90,ctr_prev90,avg_position_prev90,gsc_available_days_prev90,impressions_per_day_march,impressions_per_day_prev90,ctr_change,position_change,impressions_per_day_change,has_prev90_history
impressions_march,1.00,0.03,-0.09,0.15,0.71,0.03,-0.09,0.08,1.00,0.79,-0.01,-0.03,0.73,0.05
ctr_march,0.03,1.00,-0.19,-0.05,0.06,0.35,-0.16,-0.07,0.03,0.07,0.19,-0.09,-0.02,-0.08
avg_position_march,-0.09,-0.19,1.00,-0.12,-0.14,-0.07,0.79,-0.05,-0.09,-0.16,-0.03,0.55,0.02,0.07
gsc_available_days_march,0.15,-0.05,-0.12,1.00,0.13,-0.01,-0.21,0.33,0.13,0.15,0.00,-0.09,0.07,0.59
impressions_prev90,0.71,0.06,-0.14,0.13,1.00,0.01,-0.14,0.24,0.71,0.90,0.02,-0.05,0.14,NaN
ctr_prev90,0.03,0.35,-0.07,-0.01,0.01,1.00,-0.07,-0.08,0.03,0.03,-0.86,-0.02,0.02,NaN
avg_position_prev90,-0.09,-0.16,0.79,-0.21,-0.14,-0.07,1.00,-0.08,-0.09,-0.15,-0.01,-0.08,0.02,NaN
gsc_available_days_prev90,0.08,-0.07,-0.05,0.33,0.24,-0.08,-0.08,1.00,0.08,0.08,0.05,-0.02,0.03,0.41
impressions_per_day_march,1.00,0.03,-0.09,0.13,0.71,0.03,-0.09,0.08,1.00,0.78,-0.01,-0.03,0.73,0.04
impressions_per_day_prev90,0.79,0.07,-0.16,0.15,0.90,0.03,-0.15,0.08,0.78,1.00,0.01,-0.05,0.15,NaN


In [42]:
import numpy as np

corr_matrix = model_with_history[candidate_features].corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper_triangle.stack()
    .sort_values(ascending=False)
)

high_corr_pairs.head(15)

impressions_march          impressions_per_day_march     0.998972
impressions_prev90         impressions_per_day_prev90    0.898267
ctr_prev90                 ctr_change                    0.855857
avg_position_march         avg_position_prev90           0.787448
impressions_march          impressions_per_day_prev90    0.786718
impressions_per_day_march  impressions_per_day_prev90    0.782382
                           impressions_per_day_change    0.733466
impressions_march          impressions_per_day_change    0.728111
                           impressions_prev90            0.708302
impressions_prev90         impressions_per_day_march     0.705958
gsc_available_days_march   has_prev90_history            0.588505
avg_position_march         position_change               0.552780
gsc_available_days_prev90  has_prev90_history            0.409187
ctr_march                  ctr_prev90                    0.345276
gsc_available_days_march   gsc_available_days_prev90     0.334313
dtype: flo

In [43]:
final_features = [
    # Current state
    "impressions_per_day_march",
    "ctr_march",
    "avg_position_march",

    # Historical state
    "impressions_per_day_prev90",
    "ctr_prev90",
    "avg_position_prev90",

    # Trend
    "ctr_change",
    "position_change",
    "impressions_per_day_change",

    # Historical data coverage
    "gsc_available_days_prev90",
    "has_prev90_history"
]

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

The goal of this model is to rank content by future refresh opportunity rather than only predict a class label. I will start with Logistic Regression because it provides an interpretable probability score that can be used to rank content.

The model uses only information available by the end of March 2026. March features represent the current state of each content item, while the non-overlapping previous 90-day window (December 2025–February 2026) provides historical context and trend features. April data is used only to define the future opportunity outcome.

The initial feature set includes current performance, historical performance, and changes in CTR, average position, and daily impressions. Highly redundant raw impression features were excluded after the feature correlation audit.

Because Logistic Regression is sensitive to feature scale, numeric features will be standardized inside the modeling pipeline after the train/test split. Missing historical values will also be handled inside the pipeline to avoid using test-set information during preprocessing.

Since the business question is "which content should be refreshed first?", predicted probabilities will be used as ranking scores and the model will be evaluated with Precision@K against the Week-4 baseline on the same held-out test set.

In [44]:
final_features = [
    # Current state
    "impressions_per_day_march",
    "ctr_march",
    "avg_position_march",

    # Historical state
    "impressions_per_day_prev90",
    "ctr_prev90",
    "avg_position_prev90",

    # Trend
    "ctr_change",
    "position_change",
    "impressions_per_day_change",

    # Historical data coverage
    "gsc_available_days_prev90",
    "has_prev90_history",
]

target = "opportunity"

print("Number of features:", len(final_features))
print("Target:", target)
print("Rows:", len(model_with_history))
print("Positive rate:", model_with_history[target].mean())

Number of features: 11
Target: opportunity
Rows: 95633
Positive rate: 0.12833436156975103


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [45]:
client_audit = (
    model_with_history
    .groupby("client_hash_id")
    .agg(
        n_contents=("content_hash_id", "nunique"),
        opportunity_rate=("opportunity", "mean"),
        median_ctr_march=("ctr_march", "median"),
        median_position_march=("avg_position_march", "median"),
        median_impressions_per_day_march=(
            "impressions_per_day_march", "median"
        )
    )
    .reset_index()
)

client_audit[
    [
        "n_contents",
        "opportunity_rate",
        "median_ctr_march",
        "median_position_march",
        "median_impressions_per_day_march"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
n_contents,37.0,2584.675676,4989.960282,1.000000,17.000000,500.000000,2258.000000,22083.000000
opportunity_rate,37.0,0.065174,0.057341,0.000000,0.008850,0.071429,0.090909,0.217022
median_ctr_march,37.0,0.002400,0.004196,0.000000,0.000000,0.001160,0.002685,0.024024
median_position_march,37.0,12.510864,10.283008,3.879219,7.029134,8.760438,11.961207,57.080808
median_impressions_per_day_march,37.0,17.326356,15.582907,2.000000,6.010870,10.677419,23.161290,66.354839


In [46]:
X = model_with_history[final_features].copy()
y = model_with_history["opportunity"].copy()
groups = model_with_history["client_hash_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())
print("Overall positive rate:", y.mean())

X shape: (95633, 11)
y shape: (95633,)
Number of clients: 37
Overall positive rate: 0.12833436156975103


In [47]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

In [48]:
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

In [49]:
train_clients = set(groups_train)
test_clients = set(groups_test)

client_overlap = train_clients.intersection(test_clients)

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

Train clients: 29
Test clients: 8
Client overlap: 0


In [50]:
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print(
    "Train row share:",
    round(len(X_train) / len(X), 4)
)

print(
    "Test row share:",
    round(len(X_test) / len(X), 4)
)

Train rows: 82508
Test rows: 13125
Train row share: 0.8628
Test row share: 0.1372


In [51]:
print(
    "Overall positive rate:",
    round(y.mean(), 4)
)

print(
    "Train positive rate:",
    round(y_train.mean(), 4)
)

print(
    "Test positive rate:",
    round(y_test.mean(), 4)
)

Overall positive rate: 0.1283
Train positive rate: 0.1356
Test positive rate: 0.0827


In [52]:
print("=== SPLIT CHECK ===")

print("\nRows")
print("Train:", len(X_train))
print("Test :", len(X_test))

print("\nRow share")
print("Train:", f"{len(X_train) / len(X):.2%}")
print("Test :", f"{len(X_test) / len(X):.2%}")

print("\nClients")
print("Train:", groups_train.nunique())
print("Test :", groups_test.nunique())
print("Overlap:", len(set(groups_train) & set(groups_test)))

print("\nPositive rate")
print("Overall:", f"{y.mean():.2%}")
print("Train  :", f"{y_train.mean():.2%}")
print("Test   :", f"{y_test.mean():.2%}")

=== SPLIT CHECK ===

Rows
Train: 82508
Test : 13125

Row share
Train: 86.28%
Test : 13.72%

Clients
Train: 29
Test : 8
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 13.56%
Test   : 8.27%


In [53]:
test_client_audit = (
    model_with_history.iloc[test_idx]
    .groupby("client_hash_id")
    .agg(
        n_contents=("content_hash_id", "nunique"),
        n_opportunities=("opportunity", "sum"),
        opportunity_rate=("opportunity", "mean")
    )
    .sort_values("n_contents", ascending=False)
)

test_client_audit

,n_contents,n_opportunities,opportunity_rate
client_hash_id,,,
client_fef1a8f436438636,7737,413,0.053380
client_a80fca3f171ed1de,2816,461,0.163707
client_1a730cb2640a1abf,1088,82,0.075368
client_65de48885f4ef01b,500,36,0.072000
client_0fa64a184f18a4a0,438,51,0.116438
client_b10cb2997d0c7c86,326,40,0.122699
client_3ffa76342f366962,113,1,0.008850
client_cd12bcfd98942aa1,107,1,0.009346


In [54]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx_sg, test_idx_sg) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    train_groups_sg = groups.iloc[train_idx_sg]
    test_groups_sg = groups.iloc[test_idx_sg]

    print(f"=== FOLD {fold} ===")

    print("Rows")
    print("Train:", len(train_idx_sg))
    print("Test :", len(test_idx_sg))

    print("\nRow share")
    print("Train:", f"{len(train_idx_sg) / len(X):.2%}")
    print("Test :", f"{len(test_idx_sg) / len(X):.2%}")

    print("\nClients")
    print("Train:", train_groups_sg.nunique())
    print("Test :", test_groups_sg.nunique())

    print(
        "Overlap:",
        len(set(train_groups_sg) & set(test_groups_sg))
    )

    print("\nPositive rate")
    print("Overall:", f"{y.mean():.2%}")
    print("Train  :", f"{y.iloc[train_idx_sg].mean():.2%}")
    print("Test   :", f"{y.iloc[test_idx_sg].mean():.2%}")

    print("\n")

=== FOLD 1 ===
Rows
Train: 73550
Test : 22083

Row share
Train: 76.91%
Test : 23.09%

Clients
Train: 36
Test : 1
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 10.87%
Test   : 19.37%


=== FOLD 2 ===
Rows
Train: 77628
Test : 18005

Row share
Train: 81.17%
Test : 18.83%

Clients
Train: 31
Test : 6
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 14.22%
Test   : 6.88%


=== FOLD 3 ===
Rows
Train: 76228
Test : 19405

Row share
Train: 79.71%
Test : 20.29%

Clients
Train: 22
Test : 15
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 11.33%
Test   : 18.76%


=== FOLD 4 ===
Rows
Train: 77655
Test : 17978

Row share
Train: 81.20%
Test : 18.80%

Clients
Train: 27
Test : 10
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 13.91%
Test   : 8.18%


=== FOLD 5 ===
Rows
Train: 77471
Test : 18162

Row share
Train: 81.01%
Test : 18.99%

Clients
Train: 32
Test : 5
Overlap: 0

Positive rate
Overall: 12.83%
Train  : 13.71%
Test   : 9.07%




## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [55]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

logreg_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accou

In [56]:
import numpy as np

def precision_at_k(y_true, y_score, k):
    order = np.argsort(y_score)[::-1]
    top_k_idx = order[:k]

    top_k_true = np.asarray(y_true)[top_k_idx]

    return top_k_true.mean()

In [57]:
from sklearn.base import clone

fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    # 1. Bu fold'un train/test verilerini oluştur
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    # 2. Her fold için temiz model oluştur
    fold_model = clone(logreg_pipeline)

    # 3. Modeli bu fold'un train verisinde eğit
    fold_model.fit(X_train, y_train)

    # 4. Test content'leri için probability üret
    y_prob = fold_model.predict_proba(X_test)[:, 1]

    # 5. Ranking performansını hesapla
    p20 = precision_at_k(y_test, y_prob, 20)
    p50 = precision_at_k(y_test, y_prob, 50)

    # 6. Sonuçları sakla
    fold_results.append({
        "fold": fold,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "test_positive_rate": y_test.mean(),
        "p20": p20,
        "p50": p50
    })

    # 7. Ekrana yazdır
    print(f"Fold {fold}")
    print(f"Precision@20: {p20:.2%}")
    print(f"Precision@50: {p50:.2%}")
    print()

Fold 1
Precision@20: 45.00%
Precision@50: 32.00%

Fold 2
Precision@20: 60.00%
Precision@50: 66.00%

Fold 3
Precision@20: 55.00%
Precision@50: 62.00%

Fold 4
Precision@20: 40.00%
Precision@50: 40.00%

Fold 5
Precision@20: 45.00%
Precision@50: 58.00%



In [58]:
p20 = precision_at_k(y_test, y_prob, 20)
p50 = precision_at_k(y_test, y_prob, 50)

In [59]:
print(f"Fold {fold}")
print(f"Precision@20: {p20:.2%}")
print(f"Precision@50: {p50:.2%}")
print()

Fold 5
Precision@20: 45.00%
Precision@50: 58.00%



In [60]:
import pandas as pd

baseline_path = "/Users/egegulunay/PROJECTS/FlyRank_AI_Internship/work/notebooks/work/outputs/baseline_action_score.csv"


baseline_df = pd.read_csv(baseline_path)

print("Baseline rows:", len(baseline_df))
print()
print("Columns:")
print(baseline_df.columns.tolist())

baseline_df.head()

Baseline rows: 14997

Columns:
['rank', 'client_hash_id', 'content_hash_id', 'action', 'reason_code', 'action_score', 'avg_position_march', 'impressions_march', 'ctr_march', 'ctr_threshold', 'ctr_gap']


,rank,client_hash_id,content_hash_id,action,reason_code,action_score,avg_position_march,impressions_march,ctr_march,ctr_threshold,ctr_gap
0,1,client_23a62021009f63c4,content_44f34c0a90047651,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,305.819876,0.665877,212404.0,0.000113,0.001553,0.001440
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,208.602484,2.693038,134984.0,0.000007,0.001553,0.001545
2,3,client_73cda7b4e4f265ea,content_fec55986a1868d62,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,191.663043,0.308426,124075.0,0.000008,0.001553,0.001545
3,4,client_62f4a7e64f5e0096,content_34a70fea29d15f24,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,183.295886,3.166132,143019.0,0.000301,0.001582,0.001282
4,5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,CTR_FIX,LOW_CTR_HIGH_VISIBILITY,155.227848,9.735658,107584.0,0.000139,0.001582,0.001443


In [61]:
baseline_scores = baseline_df[
    ["client_hash_id", "content_hash_id", "action_score"]
].copy()

evaluation_data = model_with_history.merge(
    baseline_scores,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

evaluation_data["baseline_candidate"] = (
    evaluation_data["action_score"].notna().astype(int)
)

print("Modeling population:", len(model_with_history))
print("After join:", len(evaluation_data))

print(
    "Baseline candidates in modeling population:",
    evaluation_data["baseline_candidate"].sum()
)

print(
    "Baseline candidate rate:",
    f'{evaluation_data["baseline_candidate"].mean():.2%}'
)

Modeling population: 95633
After join: 95633
Baseline candidates in modeling population: 14760
Baseline candidate rate: 15.43%


In [62]:
baseline_fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    # Logistic Regression'da kullandığımız aynı test fold'u
    fold_test = evaluation_data.iloc[test_idx].copy()

    # Baseline'ın candidate seçtiklerini al
    baseline_ranked = (
        fold_test[fold_test["baseline_candidate"] == 1]
        .sort_values("action_score", ascending=False)
    )

    # Top 20 ve Top 50
    top20 = baseline_ranked.head(20)
    top50 = baseline_ranked.head(50)

    p20 = top20["opportunity"].mean()
    p50 = top50["opportunity"].mean()

    baseline_fold_results.append({
        "fold": fold,
        "n_candidates": len(baseline_ranked),
        "p20": p20,
        "p50": p50
    })

    print(f"Fold {fold}")
    print(f"Baseline candidates: {len(baseline_ranked)}")
    print(f"Precision@20: {p20:.2%}")
    print(f"Precision@50: {p50:.2%}")
    print()

Fold 1
Baseline candidates: 4699
Precision@20: 90.00%
Precision@50: 90.00%

Fold 2
Baseline candidates: 1684
Precision@20: 85.00%
Precision@50: 86.00%

Fold 3
Baseline candidates: 4597
Precision@20: 85.00%
Precision@50: 84.00%

Fold 4
Baseline candidates: 1540
Precision@20: 60.00%
Precision@50: 72.00%

Fold 5
Baseline candidates: 2240
Precision@20: 85.00%
Precision@50: 74.00%



In [63]:
logreg_results_df = pd.DataFrame(fold_results)
baseline_results_df = pd.DataFrame(baseline_fold_results)

comparison_df = logreg_results_df[
    ["fold", "test_positive_rate", "p20", "p50"]
].merge(
    baseline_results_df[
        ["fold", "n_candidates", "p20", "p50"]
    ],
    on="fold",
    suffixes=("_logreg", "_baseline")
)

comparison_df

,fold,test_positive_rate,p20_logreg,p50_logreg,n_candidates,p20_baseline,p50_baseline
0,1,0.193678,0.45,0.32,4699,0.90,0.90
1,2,0.068759,0.60,0.66,1684,0.85,0.86
2,3,0.187581,0.55,0.62,4597,0.85,0.84
3,4,0.081767,0.40,0.40,1540,0.60,0.72
4,5,0.090739,0.45,0.58,2240,0.85,0.74


In [64]:
summary_df = pd.DataFrame({
    "Method": ["Logistic Regression", "Week-4 Baseline"],
    "Mean Precision@20": [
        comparison_df["p20_logreg"].mean(),
        comparison_df["p20_baseline"].mean()
    ],
    "Mean Precision@50": [
        comparison_df["p50_logreg"].mean(),
        comparison_df["p50_baseline"].mean()
    ]
})

summary_df

,Method,Mean Precision@20,Mean Precision@50
0,Logistic Regression,0.49,0.516
1,Week-4 Baseline,0.81,0.812


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
from sklearn.base import clone

oof_predictions = []

for fold, (train_idx, test_idx) in enumerate(
    sgkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    # Fresh pipeline for this fold
    fold_model = clone(logreg_pipeline)

    # Train only on this fold's training clients
    fold_model.fit(X_train, y_train)

    # Opportunity probability for unseen test clients
    y_prob = fold_model.predict_proba(X_test)[:, 1]

    fold_output = model_with_history.iloc[test_idx][
        ["client_hash_id", "content_hash_id", "opportunity"]
    ].copy()

    fold_output["fold"] = fold
    fold_output["logreg_probability"] = y_prob

    oof_predictions.append(fold_output)

oof_predictions = pd.concat(
    oof_predictions,
    ignore_index=True
)

print("OOF rows:", len(oof_predictions)) #OOF = Out-of-Fold.
print("Unique contents:", oof_predictions["content_hash_id"].nunique())

oof_predictions.head()

OOF rows: 95633
Unique contents: 95633


,client_hash_id,content_hash_id,opportunity,fold,logreg_probability
0,client_73cda7b4e4f265ea,content_98e40d67b1940c1a,0,1,0.000109
1,client_73cda7b4e4f265ea,content_a61635ecfbb9106f,1,1,0.249448
2,client_73cda7b4e4f265ea,content_9029d0f489c63b17,0,1,0.166783
3,client_73cda7b4e4f265ea,content_90c218d081a85755,0,1,0.095681
4,client_73cda7b4e4f265ea,content_45eb191559f58a77,0,1,0.305033


In [66]:
error_analysis = oof_predictions.merge(
    baseline_df[
        ["client_hash_id", "content_hash_id", "action_score"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

error_analysis["baseline_candidate"] = (
    error_analysis["action_score"].notna().astype(int)
)

print("Rows:", len(error_analysis))
print(
    "Baseline candidates:",
    error_analysis["baseline_candidate"].sum()
)

error_analysis.head()

Rows: 95633
Baseline candidates: 14760


,client_hash_id,content_hash_id,opportunity,fold,logreg_probability,action_score,baseline_candidate
0,client_73cda7b4e4f265ea,content_98e40d67b1940c1a,0,1,0.000109,NaN,0
1,client_73cda7b4e4f265ea,content_a61635ecfbb9106f,1,1,0.249448,NaN,0
2,client_73cda7b4e4f265ea,content_9029d0f489c63b17,0,1,0.166783,NaN,0
3,client_73cda7b4e4f265ea,content_90c218d081a85755,0,1,0.095681,NaN,0
4,client_73cda7b4e4f265ea,content_45eb191559f58a77,0,1,0.305033,0.841772,1


In [67]:
error_analysis["logreg_rank"] = (
    error_analysis
    .groupby("fold")["logreg_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

In [68]:
error_analysis["baseline_rank"] = (
    error_analysis[
        error_analysis["baseline_candidate"] == 1
    ]
    .groupby("fold")["action_score"]
    .rank(method="first", ascending=False)
)

In [69]:
comparison_columns = [
    "client_hash_id",
    "content_hash_id",
    "fold",
    "opportunity",
    "logreg_probability",
    "logreg_rank",
    "action_score",
    "baseline_rank",
    "baseline_candidate"
]

error_analysis[
    comparison_columns
].sort_values(
    ["fold", "logreg_rank"]
).head(20)

,client_hash_id,content_hash_id,fold,opportunity,logreg_probability,logreg_rank,action_score,baseline_rank,baseline_candidate
15532,client_73cda7b4e4f265ea,content_4f47c103b3da99c3,1,1,0.997263,1,27.705696,52.0,1
13040,client_73cda7b4e4f265ea,content_fec55986a1868d62,1,1,0.988039,2,191.663043,2.0,1
13387,client_73cda7b4e4f265ea,content_8e1334d6356668e3,1,1,0.985303,3,208.602484,1.0,1
244,client_73cda7b4e4f265ea,content_e241d6415ac9e534,1,0,0.984195,4,NaN,NaN,0
6439,client_73cda7b4e4f265ea,content_471d9cabce329a66,1,0,0.984084,5,NaN,NaN,0
11781,client_73cda7b4e4f265ea,content_b6337ff9d230a1f7,1,1,0.982494,6,NaN,NaN,0
5120,client_73cda7b4e4f265ea,content_fd2117c2c6790e4b,1,0,0.978481,7,NaN,NaN,0
12793,client_73cda7b4e4f265ea,content_e7af1de8bf2d8e41,1,1,0.977191,8,2.284161,1653.0,1
3142,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,1,1,0.964610,9,37.281056,26.0,1
11187,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1,0,0.958533,10,129.177019,3.0,1


In [70]:
# ---------------------------------------------------------
# Error analysis: Baseline vs Logistic Regression at Top-50
# ---------------------------------------------------------

analysis = error_analysis.copy()

# 1. Her yöntemin Top-50'ye aldığı content'leri işaretle
analysis["logreg_top50"] = analysis["logreg_rank"] <= 50

analysis["baseline_top50"] = (
    analysis["baseline_candidate"].eq(1)
    & analysis["baseline_rank"].le(50)
)

# 2. Sadece gerçek opportunity=1 olanlar için
#    hangi yöntemin yakaladığını kategorize et
positive_cases = analysis[analysis["opportunity"] == 1].copy()

conditions = [
    positive_cases["logreg_top50"] & positive_cases["baseline_top50"],
    ~positive_cases["logreg_top50"] & positive_cases["baseline_top50"],
    positive_cases["logreg_top50"] & ~positive_cases["baseline_top50"],
    ~positive_cases["logreg_top50"] & ~positive_cases["baseline_top50"]
]

choices = [
    "both",
    "baseline_only",
    "logreg_only",
    "neither"
]

positive_cases["comparison_group"] = np.select(
    conditions,
    choices,
    default="unknown"
)

# 3. Her kategoride kaç gerçek opportunity olduğunu göster
group_summary = (
    positive_cases["comparison_group"]
    .value_counts()
    .reindex(["both", "baseline_only", "logreg_only", "neither"])
    .fillna(0)
    .astype(int)
)

print("=== TOP-50 OPPORTUNITY COMPARISON ===")
print(group_summary)

# 4. Fold bazında da görelim
fold_summary = pd.crosstab(
    positive_cases["fold"],
    positive_cases["comparison_group"]
)

fold_summary = fold_summary.reindex(
    columns=["both", "baseline_only", "logreg_only", "neither"],
    fill_value=0
)

print("\n=== BY FOLD ===")
display(fold_summary)

# 5. İncelemek için somut örnekler
example_columns = [
    "client_hash_id",
    "content_hash_id",
    "fold",
    "opportunity",
    "logreg_probability",
    "logreg_rank",
    "action_score",
    "baseline_rank",
    "comparison_group"
]

print("\n=== BASELINE CORRECT, LOGREG MISSED ===")
display(
    positive_cases[
        positive_cases["comparison_group"] == "baseline_only"
    ][example_columns]
    .sort_values(["fold", "baseline_rank"])
    .head(10)
)

print("\n=== LOGREG CORRECT, BASELINE MISSED ===")
display(
    positive_cases[
        positive_cases["comparison_group"] == "logreg_only"
    ][example_columns]
    .sort_values(["fold", "logreg_rank"])
    .head(10)
)

=== TOP-50 OPPORTUNITY COMPARISON ===
comparison_group
both                65
baseline_only      138
logreg_only         64
neither          12006
Name: count, dtype: int64

=== BY FOLD ===


comparison_group,both,baseline_only,logreg_only,neither
fold,,,,
1,5,40,11,4221
2,20,23,13,1182
3,10,32,21,3577
4,9,27,11,1423
5,21,16,8,1603



=== BASELINE CORRECT, LOGREG MISSED ===


,client_hash_id,content_hash_id,fold,opportunity,logreg_probability,logreg_rank,action_score,baseline_rank,comparison_group
15782,client_73cda7b4e4f265ea,content_425715547c6a3ea8,1,1,0.695204,82,110.153481,4.0,baseline_only
8382,client_73cda7b4e4f265ea,content_1bb7d17cac7f6b78,1,1,0.722112,66,67.667722,5.0,baseline_only
20164,client_73cda7b4e4f265ea,content_2e021ead8d8c7cf6,1,1,0.505946,333,58.780063,6.0,baseline_only
10240,client_73cda7b4e4f265ea,content_2aac65d27ccd9f91,1,1,0.706073,73,56.838608,7.0,baseline_only
2532,client_73cda7b4e4f265ea,content_894c99c720d52854,1,1,0.573388,178,54.433544,9.0,baseline_only
6861,client_73cda7b4e4f265ea,content_d30a67f972196ca1,1,1,0.620383,132,54.066456,10.0,baseline_only
20715,client_73cda7b4e4f265ea,content_e9f2d0579387d3c3,1,1,0.722518,65,50.302215,11.0,baseline_only
16203,client_73cda7b4e4f265ea,content_198808ff9bf71188,1,1,0.646982,114,49.180380,12.0,baseline_only
2364,client_73cda7b4e4f265ea,content_beca9fe60e659478,1,1,0.731150,61,48.469937,13.0,baseline_only
19722,client_73cda7b4e4f265ea,content_40ffbbae932be2a0,1,1,0.582401,168,48.360759,14.0,baseline_only



=== LOGREG CORRECT, BASELINE MISSED ===


,client_hash_id,content_hash_id,fold,opportunity,logreg_probability,logreg_rank,action_score,baseline_rank,comparison_group
15532,client_73cda7b4e4f265ea,content_4f47c103b3da99c3,1,1,0.997263,1,27.705696,52.0,logreg_only
11781,client_73cda7b4e4f265ea,content_b6337ff9d230a1f7,1,1,0.982494,6,NaN,NaN,logreg_only
12793,client_73cda7b4e4f265ea,content_e7af1de8bf2d8e41,1,1,0.977191,8,2.284161,1653.0,logreg_only
615,client_73cda7b4e4f265ea,content_b134852a52367a67,1,1,0.955885,11,NaN,NaN,logreg_only
9670,client_73cda7b4e4f265ea,content_6328bd18c830408d,1,1,0.925806,17,NaN,NaN,logreg_only
5223,client_73cda7b4e4f265ea,content_f4895f580257c3b2,1,1,0.905524,21,9.798137,290.0,logreg_only
19049,client_73cda7b4e4f265ea,content_336f7f8d010aac37,1,1,0.885248,27,NaN,NaN,logreg_only
3813,client_73cda7b4e4f265ea,content_b9b002b4b08aa856,1,1,0.815016,39,12.283228,203.0,logreg_only
6232,client_73cda7b4e4f265ea,content_edd684ffffeb8cb3,1,1,0.814479,40,9.040373,336.0,logreg_only
20398,client_73cda7b4e4f265ea,content_f778e170b71c5e12,1,1,0.780263,44,3.229430,1158.0,logreg_only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.